In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import os

# Directory contenente i file .mat
cartella_dati = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
# Directory dove salvare il grafico
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi'
os.makedirs(directory_save, exist_ok=True)

# Inizializza una lista per raccogliere i dati DPR
dati_dpr = []

# Carica tutti i file .mat nella cartella
for nome_file in os.listdir(cartella_dati):
    if nome_file.endswith('.mat'):
        percorso_file = os.path.join(cartella_dati, nome_file)
        print(f"Caricamento del file: {percorso_file}")
        try:
            data = scipy.io.loadmat(percorso_file)
            # Prova a trovare la chiave corretta per DPR
            if 'dpr' in data:
                dpr = data['dpr']
            elif 'precipitazione' in data:
                dpr = data['precipitazione']
            else:
                print(f"Dati di precipitazione non trovati in {nome_file}.")
                continue

            # Aggiungi i dati non NaN alla lista
            valori = dpr[~np.isnan(dpr)]
            dati_dpr.extend(valori.flatten())
        except Exception as e:
            print(f"Errore nel caricamento di {nome_file}: {e}")

# Converti in array NumPy
dati_dpr = np.array(dati_dpr)

# Verifica se ci sono dati
if dati_dpr.size == 0:
    print("Nessun dato di precipitazione trovato.")
else:
    print(f"Numero totale di valori DPR: {dati_dpr.size}")

    plt.figure(figsize=(10, 6))
    plt.hist(dati_dpr, bins=np.linspace(0.1, 200, 100), color='dodgerblue', alpha=0.7, edgecolor='black')
    plt.yscale('log')  # scala log sull’asse delle frequenze
    plt.xlabel('Precipitazione (mm/h)')
    plt.ylabel('Frequenza (scala log)')
    plt.title('Istogramma DPR (scala logaritmica in y)')

    # Salva il grafico
    path_salvataggio = os.path.join(directory_save, 'istogramma_dpr.png')
    plt.savefig(path_salvataggio)
    plt.close()
    print(f"Istogramma salvato in: {path_salvataggio}")

In [ ]:


import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import os

# Definisci il percorso al file dati_uniti.mat
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'

# Carica il file .mat
print(f"Caricamento del file: {cartella_base}")

def carica_dati(cartella):
    dati = {str(i): [] for i in range(1, 12)}  # Canali 1-11
    
    for file in os.listdir(cartella):
        if file.endswith(".mat"):
            file_path = os.path.join(cartella, file)
            mat_data = scipy.io.loadmat(file_path)
            
            for key in dati.keys():
                if key in mat_data:
                    dati[key].append(mat_data[key])
    
    # Converti liste in array numpy
    for key in dati:
        if len(dati[key]) > 0:
            dati[key] = np.concatenate(dati[key], axis=None)  # Flatten
        else:
            dati[key] = np.array([])  # Vuoto se no dati
    
    return dati

dati_input = carica_dati(cartella_base)

# Estrai i dati dai canali dal 1 al 11 (i canali sono numerati da 1 a 16, e sono nel formato di stringhe)
channels = {}
for i in range(1, 12):  # Canali da 1 a 11
    key = str(i)
    if key in dati_input:
        channels[key] = dati_input[key]  # Salva i dati del canale
        print(f"Canale {key} caricato con dimensione: {dati_input[key].shape}")
    else:
        print(f"Canale {key} non trovato nel file.")
        


# Imposta la directory per salvare i grafici
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi'
os.makedirs(directory_save, exist_ok=True)  # Crea la cartella 'istogrammi' se non esiste
print(f"Directory di salvataggio: {directory_save}")

# Funzione per calcolare l'istogramma
def calcola_istogramma(dati, matrice, chiave):
    print(f"Calcolando l'istogramma per il canale {chiave}...")
    if chiave in ['1', '2', '3']:  # Larghezza bin = 0.1 per canali 1, 2, 3
        num_bins = int((25 - 0.5) / 0.1)
    else:  # Per gli altri canali, larghezza bin = 2
        num_bins = int((340 - 180) / 2)
    valori_mascherati = matrice[~np.isnan(matrice)]  # Escludi i NaN
    hist, bins = np.histogram(valori_mascherati.flatten(), bins=num_bins, density=False)
    print(f"Numero di bin: {len(bins)}")
    return hist, bins

# Ciclo su tutti i canali dal 1 all'11
for chiave, matrice in channels.items():
    print(f"Creazione istogramma per il canale {chiave}...")
    
    # Calcola istogramma
    hist, bins = calcola_istogramma(dati_input[chiave],matrice, chiave)
    
    # Filtra i bin
    condizione = bins[:-1] >= 0.5
    hist_filtered = hist[condizione]
    bins_filtered = bins[:-1][condizione]
    
    # Verifica che ci siano dati nell'istogramma
    print(f"Numero di bin filtrati: {len(bins_filtered)}")
    print(f"Primi valori dell'istogramma: {hist_filtered[:5]}")
    
    # Plot dell'istogramma
    plt.figure(figsize=(10, 6))
    plt.hist(bins_filtered, bins, weights=hist_filtered, alpha=0.7, label=f'Canale {chiave}', density=True)
    
    plt.xlabel('Temperatura di brillanza (K)' if chiave not in ['1', '2', '3'] else 'Riflettanza (%)')
    plt.ylabel('Frequenza')
    plt.title(f'Istogramma Canale {chiave}')
    
    # Salva il grafico nella cartella 'istogrammi'
    grafico_path = os.path.join(directory_save, f'istogramma_canale_{chiave}.png')
    print(f"Salvataggio grafico in: {grafico_path}")
    plt.savefig(grafico_path)
    plt.close()  # Chiude il grafico per evitare troppi grafici aperti

In [ ]:
############## RAIN no RAIN ###########

import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt

# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'
# Directory di salvataggio
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain'
os.makedirs(directory_save, exist_ok=True)

def carica_dati(cartella):
    dati = {}

    for file in os.listdir(cartella):
        if file.endswith(".mat"):
            file_path = os.path.join(cartella, file)
            mat_data = scipy.io.loadmat(file_path)
            
            for key, value in mat_data.items():
                if key.startswith('__'):  # Salta chiavi interne di MATLAB come __header__, __version__, __globals__
                    continue
                
                if key not in dati:
                    dati[key] = [value]
                else:
                    dati[key].append(value)

    # Converti liste in array numpy
    for key in dati:
        try:
            dati[key] = np.concatenate(dati[key], axis=None)  # Flatten
        except Exception as e:
            print(f"⚠️ Errore concatenando la chiave '{key}': {e}")
            dati[key] = np.array(dati[key])  # Lascia come lista di array se non concatenabile

    return dati
# Carica i dati uniti
dati_mat = carica_dati(cartella_base)
# Carica le maschere rain e no rain
mask_rain = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'maschera_rain.mat'))['maschera_rain'].flatten()
mask_norain = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'maschera_norain.mat'))['maschera_norain'].flatten()

# Seleziona i canali 1-11
canali_da_usare = [str(i) for i in range(1, 12)]

# Funzione per calcolare l'istogramma
def calcola_istogramma(dati, maschera, chiave):
    dati_mascherati = dati[maschera == 1]  # Applica maschera
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]  # Escludi NaN

    if chiave in ['1', '2', '3']:  # Larghezza bin = 0.1 per riflettanza
        bins = np.arange(0.5, 80, 0.2)
    else:  # Larghezza bin = 2 per temperatura di brillanza
        bins = np.arange(180, 340, 2)

    hist, _ = np.histogram(dati_mascherati, bins=bins, density=True)
    return hist, bins

# Creazione degli istogrammi per ogni canale
for chiave in canali_da_usare:
    if chiave not in dati_mat:
        print(f"Dati mancanti per il canale {chiave}, salto il confronto.")
        continue

    dati_canale = dati_mat[chiave].flatten()

    # Calcola gli istogrammi per rain e no rain
    hist_rain, bins = calcola_istogramma(dati_canale, mask_rain, chiave)
    hist_norain, _ = calcola_istogramma(dati_canale, mask_norain, chiave)

    # Plot
    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 40})
    # Istogramma per rain (blu)
    plt.bar(bins[:-1], hist_rain, width=np.diff(bins), alpha=0.9, color='#00f2aa', label='Rain')
    # Istogramma per no rain (verde)
    plt.bar(bins[:-1], hist_norain, width=np.diff(bins), alpha=0.8, color='#E8AEE0', label='No Rain')

    plt.xlabel('Brightness temperature (K)' if chiave not in ['1', '2', '3'] else 'Reflectance (%)', labelpad=15, fontsize=50)
    plt.ylabel('Density', labelpad=15, fontsize=50)
    #plt.title(f'Rain vs No Rain Histogram – Channel {chiave}')
    plt.legend(fontsize=40)
    plt.tight_layout()

    # Salvataggio
    grafico_path = os.path.join(directory_save, f'confronto_rain_norain_canale_{chiave}.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"Grafico salvato: {grafico_path}")

In [2]:
############## RAIN no RAIN NORM ###########

import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt

# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'
# Directory di salvataggio
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm'
os.makedirs(directory_save, exist_ok=True)

def carica_dati(cartella):
    dati = {}

    for file in os.listdir(cartella):
        if file.endswith(".mat"):
            file_path = os.path.join(cartella, file)
            mat_data = scipy.io.loadmat(file_path)
            
            for key, value in mat_data.items():
                if key.startswith('__'):  # Salta chiavi interne di MATLAB come __header__, __version__, __globals__
                    continue
                
                if key not in dati:
                    dati[key] = [value]
                else:
                    dati[key].append(value)

    # Converti liste in array numpy
    for key in dati:
        try:
            dati[key] = np.concatenate(dati[key], axis=None)  # Flatten
        except Exception as e:
            print(f"⚠️ Errore concatenando la chiave '{key}': {e}")
            dati[key] = np.array(dati[key])  # Lascia come lista di array se non concatenabile

    return dati
# Carica i dati uniti
dati_mat = carica_dati(cartella_base)
# Carica le maschere rain e no rain
mask_rain = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'maschera_rain.mat'))['maschera_rain'].flatten()
mask_norain = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'maschera_norain.mat'))['maschera_norain'].flatten()

# Seleziona i canali 1-11
canali_da_usare = [str(i) for i in range(1, 12)]

# Funzione per calcolare l'istogramma normalizzato sul totale
def calcola_istogramma(dati, maschera, chiave, totale_pixel):
    dati_mascherati = dati[maschera == 1]
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]  # Escludi NaN

    # Definizione dei bin
    if chiave in ['1', '2', '3']:  # canali di riflettanza
        bins = np.arange(0.5, 80, 0.2)
    else:  # canali di temperatura di brillanza
        bins = np.arange(180, 340, 2)

    # Calcolo istogramma (conteggi grezzi)
    hist, _ = np.histogram(dati_mascherati, bins=bins, density=False)

    # 🔹 Normalizzazione rispetto al totale dei pixel validi complessivi
    hist = hist / totale_pixel

    return hist, bins


# Creazione degli istogrammi per ogni canale
for chiave in canali_da_usare:
    if chiave not in dati_mat:
        print(f"Dati mancanti per il canale {chiave}, salto il confronto.")
        continue

    dati_canale = dati_mat[chiave].flatten()

    # 🔹 Calcolo del totale dei pixel validi (escludendo i NaN)
    totale_pixel = np.sum(~np.isnan(dati_canale))

    # Calcola gli istogrammi per rain e no rain
    hist_rain, bins = calcola_istogramma(dati_canale, mask_rain, chiave, totale_pixel)
    hist_norain, _ = calcola_istogramma(dati_canale, mask_norain, chiave, totale_pixel)

    # Plot
    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 40})

    plt.bar(bins[:-1], hist_rain, width=np.diff(bins), alpha=0.9, color='#00f2aa', label='Rain')
    plt.bar(bins[:-1], hist_norain, width=np.diff(bins), alpha=0.8, color='#E8AEE0', label='No Rain')

    plt.xlabel('Brightness temperature (K)' if chiave not in ['1', '2', '3'] else 'Reflectance (%)',
               labelpad=15, fontsize=50)
    plt.ylabel('Fraction of total pixels', labelpad=15, fontsize=50)
    plt.legend(fontsize=40)
    plt.tight_layout()

    # Salvataggio
    grafico_path = os.path.join(directory_save, f'confronto_rain_norain_canale_{chiave}.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"Grafico salvato: {grafico_path}")

Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_1.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_2.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_3.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_4.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_5.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_6.png
Grafico salvato: C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_rain_norain_norm\confronto_rain_norain_canale_7.png
Grafico salvato: C:/Users/user/Documents/

In [ ]:
################# ISTOGRAMMI GIORNO-NOTTE ##################

import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt

# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_giorno_notte'

# Crea la cartella di salvataggio se non esiste
os.makedirs(directory_save, exist_ok=True)

# Funzione per ordinare i file in base a DOY e TIME nel nome file
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')

# Funzione per caricare i dati dei canali da una cartella, con file ordinati
def carica_dati(cartella):
    dati = {str(i): [] for i in range(1, 12)}  # Canali 1-11
    
    # Lista file ordinati per DOY e TIME
    files_ordinati = sorted(
        [f for f in os.listdir(cartella) if f.endswith(".mat")],
        key=estrai_doy_time
    )
    
    for file in files_ordinati:
        file_path = os.path.join(cartella, file)
        mat_data = scipy.io.loadmat(file_path)
        
        for key in dati.keys():
            if key in mat_data:
                dati[key].append(mat_data[key])
    
    # Converti liste in array numpy concatenati e flattenati
    for key in dati:
        if len(dati[key]) > 0:
            dati[key] = np.concatenate(dati[key], axis=None)
        else:
            dati[key] = np.array([])
    
    return dati

# Funzione per caricare le maschere giorno e notte
def carica_mascheramenti():
    mask_giorno = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'giorno_maschera_3h.mat'))['giorno_maschera'].flatten()
    mask_notte = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'notte_maschera_3h.mat'))['notte_maschera'].flatten()
    return mask_giorno, mask_notte

# Funzione per calcolare l'istogramma
def calcola_istogramma(dati, maschera, chiave):
    dati_mascherati = dati[maschera == 1]  # Applica maschera
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]  # Escludi NaN
    
    if chiave in ['1', '2', '3']:  # Larghezza bin = 0.1
        bins = np.arange(0.5, 55, 0.2)
    else:  # Larghezza bin = 2
        bins = np.arange(180, 340, 2)
    
    hist, _ = np.histogram(dati_mascherati, bins=bins, density=True)
    return hist, bins

# Carica le maschere giorno e notte
mask_giorno, mask_notte = carica_mascheramenti()

# Carica i dati dalla cartella di INPUT_1
dati_input = carica_dati(cartella_base)

# Creazione degli istogrammi per ogni canale
for chiave in dati_input.keys():
    if dati_input[chiave].size == 0:
        print(f"Dati mancanti per il canale {chiave}, salto il confronto.")
        continue
    
    # Calcola gli istogrammi per il giorno e la notte
    hist_giorno, bins = calcola_istogramma(dati_input[chiave], mask_giorno, chiave)
    hist_notte, _ = calcola_istogramma(dati_input[chiave], mask_notte, chiave)
    
    # Plot
    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 32})
    plt.bar(bins[:-1], hist_giorno, width=np.diff(bins), alpha=0.7, color='#FFD700', label='Day')
    plt.bar(bins[:-1], hist_notte, width=np.diff(bins), alpha=0.5, color='#00BFFF', label='Night')
    
    plt.xlabel('Brightness temperature (K)' if chiave not in ['1', '2', '3'] else 'Reflectance (%)',  labelpad =15, fontsize=40)
    plt.ylabel('Density',labelpad =15, fontsize=40)
    #plt.title(f'ay vs Night - Channel {chiave}')
    plt.legend(fontsize=28)
    plt.tight_layout()
    
    # Salvataggio
    grafico_path = os.path.join(directory_save, f'confronto_canale_{chiave}_3h.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"Grafico salvato: {grafico_path}")


In [ ]:
############# istogrammi stagioni ############
import re
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

def calcola_istogramma(dati, maschera, chiave):
    dati_mascherati = dati[maschera == 1]  # Applica maschera
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]  # Escludi NaN
    
    if chiave in ['1', '2', '3']:  # Larghezza bin = 0.1
        bins = np.arange(0.5, 55, 0.2)
    else:  # Larghezza bin = 2
        bins = np.arange(180, 340, 2)
    
    hist, _ = np.histogram(dati_mascherati, bins=bins, density=True)
    return hist, bins

# Directory dei dati
data_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1"
mask_directory = "C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE"
output_hist_dir = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_estateinverno'

# Creo la cartella per salvare gli istogrammi se non esiste
os.makedirs(output_hist_dir, exist_ok=True)

# Carichiamo la maschera stagionalità
mask_path = os.path.join(mask_directory, 'stagionalita_maschera_estate_inverno.mat')
mask_data = loadmat(mask_path)
stagionalita_maschera = mask_data['stagionalita_maschera'].flatten()

# === FUNZIONE DI ORDINAMENTO ===
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')

# === CARICAMENTO FILE ORDINATO ===
mat_files = sorted(
    [f for f in os.listdir(data_directory) if f.endswith(".mat")],
    key=estrai_doy_time
)

if not mat_files:
    print("⚠️ Nessun file .mat trovato!")
    exit()

print(f"📂 Trovati {len(mat_files)} file da elaborare.")
# Carichiamo tutti i dati dei canali (dalla chiave 1 alla chiave 11)
dati_canali = [[] for _ in range(11)]  # 11 canali (1-11)

# Analizziamo ogni file .mat
for file_name in mat_files:
    file_path = os.path.join(data_directory, file_name)
    try:
        dati = loadmat(file_path)
        for canale_idx in range(1, 12):  # Canali da 1 a 11
            if str(canale_idx) in dati:
                dati_canali[canale_idx - 1].append(dati[str(canale_idx)].flatten())
            else:
                print(f"⚠️ File {file_name} manca canale {canale_idx}")
    except Exception as e:
        print(f"❌ Errore caricando {file_name}: {e}")

# Concatenazione dei dati dei canali, solo se non sono vuoti
dati_canali = [np.concatenate(dati_list) if dati_list else np.array([]) for dati_list in dati_canali]

# Verifica la dimensione dei dati e della maschera
totale_punti_mask = len(stagionalita_maschera)

# Controlliamo che tutte le concatenazioni siano corrette
for canale_idx, valori_canale in enumerate(dati_canali):
    if valori_canale.size != totale_punti_mask:
        print(f"⚠️ Il canale {canale_idx+1} ha una dimensione errata! "
              f"Dimensione canale: {valori_canale.size}, Dimensione maschera: {totale_punti_mask}")
    else:
        print(f"✅ Il canale {canale_idx+1} ha la stessa dimensione della maschera.")

if any(valori_canale.size != totale_punti_mask for valori_canale in dati_canali):
    raise ValueError("❗ Dati e maschera NON hanno la stessa dimensione! Controlla!")

# Ora possiamo applicare la maschera e calcolare istogrammi
for canale_idx, valori_canale in enumerate(dati_canali):
    if valori_canale.size == 0:
        print(f"⏩ Salto canale {canale_idx+1} perché vuoto.")
        continue

    # Separiamo estate (0) e inverno (1)
    maschera_estate = (stagionalita_maschera == 0).astype(int)
    maschera_inverno = (stagionalita_maschera == 1).astype(int)

    # Calcolo istogramma con la funzione per estate e inverno
    hist_estate, bins_estate = calcola_istogramma(valori_canale, maschera_estate, str(canale_idx+1))
    hist_inverno, bins_inverno = calcola_istogramma(valori_canale, maschera_inverno, str(canale_idx+1))

    # Plotto istogrammi normalizzati (densità)
    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 32})
    width_estate = np.diff(bins_estate)[0]
    width_inverno = np.diff(bins_inverno)[0]

    # Normalizzo manualmente (frequenza relativa)
    #hist_estate_norm = hist_estate / np.sum(hist_estate)
    #hist_inverno_norm = hist_inverno / np.sum(hist_inverno)

    plt.bar(bins_estate[:-1], hist_estate, width=width_estate, color='#FFA500', alpha=0.6, label='Summer')  # corallo
    plt.bar(bins_inverno[:-1], hist_inverno, width=width_inverno, color='#0055FF', alpha=0.5, label='Winter')  # azzurro

    #plt.title(f"Summer vs Winter Histogram - Channel {canale_idx+1}")
    plt.xlabel('Reflectance (%)' if chiave in ['1', '2', '3'] else 'Brightness temperature (K)', labelpad =15, fontsize=40)
    plt.ylabel('Density', labelpad=15, fontsize=40)
    plt.legend(fontsize=28)
    #plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Salvo l'immagine
    filename = f"istogramma_estate_inverno_canale_{canale_idx+1}.png"
    save_path = os.path.join(output_hist_dir, filename)
    plt.savefig(save_path)
    plt.close()  # Chiudo la figura per liberare memoria

    print(f"✅ Istogramma canale {canale_idx+1} salvato in {save_path}")


In [ ]:
############ Istogrammi mare e terra #########

import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import re
# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'

# Directory di salvataggio
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_mare_terra'
os.makedirs(directory_save, exist_ok=True)

# Carica le maschere mare e terra (versione regridded)
mask_mare = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'Mare_mask_regridded.mat'))['mask'].flatten()
mask_terra = scipy.io.loadmat(os.path.join(cartella_mascheramenti, 'Terra_mask_regridded.mat'))['mask'].flatten()

# Seleziona i canali 1-11
canali_da_usare = [str(i) for i in range(1, 12)]

# Dizionario per accumulare dati concatenati per canale
dati_concatenati = {canale: [] for canale in canali_da_usare}

# Ciclo su tutti i file e concateno i dati per canale
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')  # Per evitare problemi nei file senza match

# Ordina i file per DOY e TIME estratti dal nome
file_ordinati = sorted(
    [f for f in os.listdir(cartella_base) if f.endswith('.mat')],
    key=estrai_doy_time
)

# Scorri i file in ordine e carica lat/lon
for filename in file_ordinati:
    filepath = os.path.join(cartella_base, filename)
    print(f"Caricando file: {filename}")
    dati_mat = scipy.io.loadmat(filepath)
    
    for chiave in canali_da_usare:
        if chiave in dati_mat:
            dati_canale = dati_mat[chiave].flatten()
            dati_concatenati[chiave].append(dati_canale)
        else:
            print(f"Dati mancanti per il canale {chiave} in file {filename}, salto.")
# Ora concateno i dati per ogni canale in un singolo array numpy
for chiave in canali_da_usare:
    if len(dati_concatenati[chiave]) > 0:
        dati_concatenati[chiave] = np.concatenate(dati_concatenati[chiave])
    else:
        print(f"Nessun dato trovato per il canale {chiave} in nessun file.")
        dati_concatenati[chiave] = np.array([])

# Funzione per calcolare l'istogramma
def calcola_istogramma(dati, maschera, chiave):
    dati_mascherati = dati[maschera == 1]  # Applica maschera
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]  # Escludi NaN

    if chiave in ['1', '2', '3']:  # Larghezza bin = 0.1 per riflettanza
        bins = np.arange(0.5, 55, 0.2)
    else:  # Larghezza bin = 2 per temperatura di brillanza
        bins = np.arange(180, 340, 2)

    hist, _ = np.histogram(dati_mascherati, bins=bins, density=True)
    return hist, bins

# Calcolo e plot degli istogrammi concatenati per ogni canale
for chiave in canali_da_usare:
    dati_canale = dati_concatenati[chiave]
    if dati_canale.size == 0:
        print(f"Salto canale {chiave} perché senza dati.")
        continue

    hist_mare, bins = calcola_istogramma(dati_canale, mask_mare, chiave)
    hist_terra, _ = calcola_istogramma(dati_canale, mask_terra, chiave)

    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 32})
    plt.bar(bins[:-1], hist_mare, width=np.diff(bins), alpha=0.8, color='lightblue', label='Sea')
    plt.bar(bins[:-1], hist_terra, width=np.diff(bins), alpha=0.5, color='sandybrown', label='Land')

    plt.xlabel('Reflectance (%)' if chiave in ['1', '2', '3'] else 'Brightness temperature (K)', labelpad=15, fontsize=40)
    plt.ylabel('Density', labelpad=15, fontsize=40)
    #plt.title(f'Sea vs Land Histogram - Channel {chiave}')
    plt.legend(fontsize=28)
    plt.tight_layout()
    
    grafico_path = os.path.join(directory_save, f'confronto_mare_terra_canale_{chiave}.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"Grafico salvato: {grafico_path}")

In [ ]:
################ ISTOGRAMMI DPR PER CLASSI #################

import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import re

# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_classi'
os.makedirs(directory_save, exist_ok=True)

# Funzione per ordinare i file in base a DOY e TIME nel nome file
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    else:
        return float('inf'), float('inf')

# Funzione per caricare i dati dei canali da una cartella
def carica_dati(cartella):
    dati = {str(i): [] for i in range(1, 12)}  # Canali 1-11
    dati['dpr'] = []

    files_ordinati = sorted(
        [f for f in os.listdir(cartella) if f.endswith(".mat")],
        key=estrai_doy_time
    )

    for file in files_ordinati:
        file_path = os.path.join(cartella, file)
        mat_data = scipy.io.loadmat(file_path)

        for key in dati.keys():
            if key in mat_data:
                dati[key].append(mat_data[key])

    for key in dati:
        if len(dati[key]) > 0:
            dati[key] = np.concatenate(dati[key], axis=None)
        else:
            dati[key] = np.array([])

    return dati

# Caricamento dati
print("📦 Caricamento dati dai file in OUTPUT_1...")
dati_mat = carica_dati(cartella_base)
print("✅ Dati caricati!")

# Controllo variabile DPR
if 'dpr' not in dati_mat:
    raise ValueError("❌ Errore: La variabile 'dpr' non è presente nei file!")

dpr = dati_mat['dpr'].flatten()
print(f"📌 Numero di valori di DPR: {len(dpr)}")

# Definizione classi di precipitazione
soglie_pioggia = [0, 0.1, 1, 5, 15, np.inf]
nomi_classi = ["Dry", "Light", "Moderate", "Heavy", "Extreme"]
colori = ["blue", "green", "yellow", "orange", "red"]

# Seleziona gli 11 canali
tutti_canali = [str(i) for i in range(1, 12)]

print("\n📊 Inizio elaborazione delle distribuzioni per classi di DPR...\n")

for i in tqdm(range(len(tutti_canali)), desc="Processo canali"):
    chiave = tutti_canali[i]

    if chiave not in dati_mat:
        print(f"⚠️ Canale {chiave} non trovato, salto.")
        continue

    dati_canale = dati_mat[chiave].flatten()
    print(f"📡 Canale {chiave}: {len(dati_canale)} valori, NaN: {np.isnan(dati_canale).sum()}")

    dati_validi = ~np.isnan(dpr) & ~np.isnan(dati_canale)
    dpr_validi = dpr[dati_validi]
    dati_canale = dati_canale[dati_validi]

    print(f"   ➡️ Dati validi dopo rimozione NaN: {len(dati_canale)}")

    if len(dati_canale) == 0:
        print(f"❌ Nessun dato valido per il canale {chiave}, salto.")
        continue

    mask_positivi = dati_canale >= 0
    dati_canale = dati_canale[mask_positivi]
    dpr_validi = dpr_validi[mask_positivi]

    plt.figure(figsize=(16, 12))
    plt.rcParams.update({'font.size': 32})
    print(f"\n📡 Analizzando canale {chiave}...")

    for j in range(len(soglie_pioggia) - 1):
        soglia_min = soglie_pioggia[j]
        soglia_max = soglie_pioggia[j + 1]

        mask = (dpr_validi >= soglia_min) & (dpr_validi < soglia_max)
        dati_filtro = dati_canale[mask]

        print(f"  🔹 Classe '{nomi_classi[j]}': {len(dati_filtro)} valori trovati.")

        if len(dati_filtro) > 0:
            if len(dati_filtro) > 1_000_000:
                dati_filtro = np.random.choice(dati_filtro, 500000, replace=False)
                print(f"     🔻 Troppi valori! Usati 500000 campioni casuali.")

            sns.kdeplot(dati_filtro, color=colori[j], label=nomi_classi[j], clip=(0, np.inf), linewidth=3.5)

            # Solo per i canali VIS (1-2-3): limiti assi
            if chiave in ['1', '2', '3']:
                plt.xlim(0, 60)     # Limite per riflettanza
                plt.ylim(0, 0.17)    # (Facoltativo) Limite densità
    
    #plt.rcParams.update({'font.size': 30})
    plt.xlabel('Brightness temperature (K)' if chiave not in ['1', '2', '3'] else 'Reflectance (%)', labelpad =15, fontsize=40)
    plt.ylabel('Density', fontsize=40, labelpad=15)
    #plt.title(f'Rain class distribution – Channel {chiave}',pad=20, fontsize=26)
    plt.legend(fontsize=28)
    plt.tight_layout()
    
    grafico_path = os.path.join(directory_save, f'distribuzione_classi_canale_{chiave}.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"📊 Grafico salvato: {grafico_path}")
    
    
print("\n✅ Elaborazione completata!")

In [ ]:
import os
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt

# Percorsi delle cartelle
cartella_base = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/OUTPUT_1'
cartella_mascheramenti = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/MASCHERE'
directory_save = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/istogrammi_mare_terra'
os.makedirs(directory_save, exist_ok=True)

# Carica le maschere mare e terra
mask_mare = sio.loadmat(os.path.join(cartella_mascheramenti, 'Mare_mask_regridded.mat'))['mask'].flatten()
mask_terra = sio.loadmat(os.path.join(cartella_mascheramenti, 'Terra_mask_regridded.mat'))['mask'].flatten()

# Inizializza dizionario per contenere tutti i dati caricati
dati_concatenati = {}

# Caricamento e concatenazione
print("Caricamento e concatenazione dei dati...")
for filename in os.listdir(cartella_base):
    if filename.endswith('.mat'):
        filepath = os.path.join(cartella_base, filename)
        print(f"Caricando file: {filename}")
        dati_mat = sio.loadmat(filepath)

        for chiave, valore in dati_mat.items():
            if chiave.startswith('__'):
                continue  # Salta le chiavi di sistema di MATLAB (__header__, __globals__, ecc.)

            valore = valore.flatten()

            if chiave not in dati_concatenati:
                dati_concatenati[chiave] = [valore]
            else:
                dati_concatenati[chiave].append(valore)

# Unisci tutti i dati per ogni chiave
for chiave in dati_concatenati:
    dati_concatenati[chiave] = np.concatenate(dati_concatenati[chiave])

print("Tutti i dati caricati e concatenati.")

# Funzione per calcolare l'istogramma
def calcola_istogramma(dati, maschera, chiave):
    dati_mascherati = dati[maschera == 1]
    dati_mascherati = dati_mascherati[~np.isnan(dati_mascherati)]

    if chiave in ['1', '2', '3']:  # Se il canale è uno dei primi tre (riflettanza)
        bins = np.arange(0.5, 35, 0.2)
    else:  # Altrimenti temperatura
        bins = np.arange(180, 340, 2)

    hist, _ = np.histogram(dati_mascherati, bins=bins, density=True)
    return hist, bins

# Creazione degli istogrammi
for chiave, dati in dati_concatenati.items():
    if dati.size == 0:
        print(f"Salto il canale {chiave} perché non ci sono dati.")
        continue

    if dati.shape[0] != mask_mare.shape[0]:
        print(f"Attenzione: la dimensione dei dati ({dati.shape[0]}) non combacia con la maschera ({mask_mare.shape[0]}). Skipping {chiave}.")
        continue

    # Calcola istogrammi
    hist_mare, bins = calcola_istogramma(dati, mask_mare, chiave)
    hist_terra, _ = calcola_istogramma(dati, mask_terra, chiave)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(bins[:-1], hist_mare, width=np.diff(bins), alpha=0.5, color='lightblue', label='Mare')
    plt.bar(bins[:-1], hist_terra, width=np.diff(bins), alpha=0.5, color='sandybrown', label='Terra')

    plt.xlabel('Riflettanza (%)' if chiave in ['1', '2', '3'] else 'Temperatura di brillanza (K)')
    plt.ylabel('Frequenza')
    plt.title(f'Istogramma Mare vs Terra - Canale {chiave}')
    plt.legend()

    # Salvataggio
    grafico_path = os.path.join(directory_save, f'confronto_mare_terra_canale_{chiave}.png')
    plt.savefig(grafico_path)
    plt.close()
    print(f"Grafico salvato: {grafico_path}")

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# === 1. Caricamento delle predizioni ===
output_directory_new = 'C:/Users/user/Documents/TESI/datigrezzi/DATI_grezzi_2020/Dati_ML_plus_5classi/output'

with open(os.path.join(output_directory_new, 'y_pred_RF.pickle'), 'rb') as f:
    y_pred = pickle.load(f)

# === 2. Dati da usare: canale9_tot e dpr_tot devono essere già disponibili ===
# Assicurati che questi array siano già presenti:
# - canale9_tot: valori di canale 9 (1D array)
# - dpr_tot: valori corrispondenti di DPR (1D array)

# === 3. Primo grafico: boxplot + barre dei massimi ===
plt.figure(figsize=(14, 6))

# Ordine classi desiderato
ordine_classi = ['Dry', 'Light', 'Moderate', 'Heavy', 'Intense']
dati_ordinati = [canale9_tot[np.array(y_pred) == label] for label in ordine_classi]

# Boxplot
plt.subplot(1, 2, 1)
plt.boxplot(dati_ordinati, labels=ordine_classi)
plt.title("Boxplot Canale 9 per Classe Predetta")
plt.xlabel("Classe Predetta (y_pred)")
plt.ylabel("Valori Canale 9")
plt.grid(True, linestyle='--', alpha=0.5)

# Grafico a barre con valore massimo
plt.subplot(1, 2, 2)
max_valori = [np.max(x) if len(x) > 0 else 0 for x in dati_ordinati]
plt.bar(ordine_classi, max_valori, color='skyblue')
plt.title("Valore Massimo Canale 9 per Classe Predetta")
plt.xlabel("Classe Predetta (y_pred)")
plt.ylabel("Valore massimo Canale 9")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# === 4. Secondo grafico: scatter DPR vs Canale 9 con sfumatura blu ===
dpr_bins = [0, 0.1, 1, 5, 15, np.max(dpr_tot)]
dpr_indices = np.digitize(dpr_tot, dpr_bins) - 1  # valori da 0 a 4

# Sfumature di blu (dal chiaro allo scuro)
sfumature_blu = ['#cce5ff', '#99ccff', '#66b3ff', '#3399ff', '#0066cc']
colori = [sfumature_blu[i] if 0 <= i < len(sfumature_blu) else 'black' for i in dpr_indices]

plt.figure(figsize=(10, 6))
plt.scatter(dpr_tot, canale9_tot, c=colori, alpha=0.4, s=5)
plt.xscale('log')
plt.xlabel('DPR (mm/h, scala logaritmica)')
plt.ylabel('Canale 9')
plt.title('Distribuzione Canale 9 in funzione di DPR')
plt.grid(True, which='both', linestyle='--', alpha=0.5)

# Legenda manuale
legenda = [
    mpatches.Patch(color=sfumature_blu[0], label='0–0.1 mm/h'),
    mpatches.Patch(color=sfumature_blu[1], label='0.1–1 mm/h'),
    mpatches.Patch(color=sfumature_blu[2], label='1–5 mm/h'),
    mpatches.Patch(color=sfumature_blu[3], label='5–15 mm/h'),
    mpatches.Patch(color=sfumature_blu[4], label='>15 mm/h'),
]
plt.legend(handles=legenda, title='Classi di DPR', loc='upper right')

plt.show()

In [ ]:
import os
import re
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pickle
import matplotlib.patches as mpatches

# === Parametri ===
cartella = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\OUTPUT_1'
output_directory_new = r'C:\Users\user\Documents\TESI\datigrezzi\DATI_grezzi_2020\Dati_ML_plus_5classi\output'

# Funzione per ordinare i file
def estrai_doy_time(nome_file):
    match = re.search(r"DOY(\d+)_TIME(\d+)", nome_file)
    if match:
        doy = int(match.group(1))
        time = int(match.group(2))
        return doy, time
    return float('inf'), float('inf')

# Ordina i file
file_ordinati = sorted(
    [f for f in os.listdir(cartella) if f.endswith('.mat')],
    key=estrai_doy_time
)

print(f"Numero file .mat trovati: {len(file_ordinati)}")
if len(file_ordinati) == 0:
    print("ATTENZIONE: Nessun file .mat trovato nella cartella.")
    exit()

# Carica dati canale 9 e DPR da tutti i file, concatenandoli
canale9_tot = []
dpr_tot = []

for i, file in enumerate(file_ordinati):
    print(f"\nCaricamento file {i+1}/{len(file_ordinati)}: {file}")
    data = sio.loadmat(os.path.join(cartella, file))
    print(f"Chiavi nel file: {list(data.keys())}")
    
    # Controllo se esistono le chiavi
    if '9' in data:
        canale9 = data['9']
    else:
        print("Chiave '9' NON trovata, provo con 'channel9'...")
        if 'channel9' in data:
            canale9 = data['channel9']
        else:
            print(f"ERRORE: Chiave per canale 9 non trovata nel file {file}")
            continue

    if 'dpr' in data:
        dpr = data['dpr']
    else:
        print("Chiave 'dpr' NON trovata, provo con 'DPR'...")
        if 'DPR' in data:
            dpr = data['DPR']
        else:
            print(f"ERRORE: Chiave per DPR non trovata nel file {file}")
            continue

    print(f"Dimensioni canale9: {canale9.shape}, DPR: {dpr.shape}")

    # Flatten dei dati (modifica se serve reshape differente)
    canale9 = canale9.flatten()
    dpr = dpr.flatten()
    print(f"Dimensioni dopo flatten: canale9 {canale9.shape}, DPR {dpr.shape}")

    if canale9.shape[0] != dpr.shape[0]:
        print(f"ATTENZIONE: dimensioni diverse tra canale9 e DPR in file {file}, salto file.")
        continue

    canale9_tot.append(canale9)
    dpr_tot.append(dpr)

if len(canale9_tot) == 0 or len(dpr_tot) == 0:
    print("ERRORE: Nessun dato caricato correttamente.")
    exit()

canale9_tot = np.concatenate(canale9_tot)
dpr_tot = np.concatenate(dpr_tot)
print(f"\nDimensioni totali dati concatenati: canale9_tot={canale9_tot.shape}, dpr_tot={dpr_tot.shape}")

# Carica y_pred dal pickle
pickle_path = os.path.join(output_directory_new, 'y_pred_RF.pickle')
print(f"\nCaricamento y_pred da: {pickle_path}")
with open(pickle_path, 'rb') as f:
    y_pred = pickle.load(f)

print(f"Dimensione y_pred: {len(y_pred)}")
assert len(y_pred) == len(canale9_tot), "ERRORE: y_pred e dati non sono allineati per dimensione"

# === 3. Primo grafico: boxplot + scatter ===
plt.figure(figsize=(14, 6))

# Ordine classi desiderato
ordine_classi = ['Dry', 'Light', 'Moderate', 'Heavy', 'Intense']
class_to_index = {label: i for i, label in enumerate(ordine_classi)}

# Boxplot
dati_ordinati = []
for label in ordine_classi:
    dati_classe = canale9_tot[np.array(y_pred) == label]
    print(f"Classe {label}: {len(dati_classe)} valori")
    dati_ordinati.append(dati_classe)

plt.subplot(1, 2, 1)
plt.boxplot(dati_ordinati, labels=ordine_classi)
plt.title("Boxplot Canale 9 per Classe Predetta")
plt.xlabel("Classe Predetta (y_pred)")
plt.ylabel("Valori Canale 9")
plt.grid(True, linestyle='--', alpha=0.5)

# Scatter plot: classe predetta vs canale 9
plt.subplot(1, 2, 2)
x_pred = [class_to_index[cls] for cls in y_pred if cls in class_to_index]
y_vals = [canale9_tot[i] for i, cls in enumerate(y_pred) if cls in class_to_index]
colori = ['#cce5ff', '#99ccff', '#66b3ff', '#3399ff', '#0066cc']
c = [colori[class_to_index[cls]] for cls in y_pred if cls in class_to_index]

plt.scatter(x_pred, y_vals, c=c, alpha=0.4, s=5)
plt.xticks(ticks=range(len(ordine_classi)), labels=ordine_classi)
plt.title("Scatter: Classe Predetta vs Canale 9")
plt.xlabel("Classe Predetta (y_pred)")
plt.ylabel("Valori Canale 9")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# === 4. Secondo grafico: scatter DPR vs Canale 9 con sfumatura blu ===
dpr_bins = [0, 0.1, 1, 5, 15, np.max(dpr_tot)]
dpr_indices = np.digitize(dpr_tot, dpr_bins) - 1  # valori da 0 a 4

# Sfumature di blu (dal chiaro allo scuro)
sfumature_blu = ['#cce5ff', '#99ccff', '#66b3ff', '#3399ff', '#0066cc']
colori_dpr = [sfumature_blu[i] if 0 <= i < len(sfumature_blu) else 'black' for i in dpr_indices]

plt.figure(figsize=(10, 6))
plt.scatter(dpr_tot, canale9_tot, c=colori_dpr, alpha=0.4, s=5)
plt.xscale('log')
plt.xlabel('DPR (mm/h, scala logaritmica)')
plt.ylabel('Canale 9')
plt.title('Distribuzione Canale 9 in funzione di DPR')
plt.grid(True, which='both', linestyle='--', alpha=0.5)

# Legenda manuale
legenda = [
    mpatches.Patch(color=sfumature_blu[0], label='0–0.1 mm/h'),
    mpatches.Patch(color=sfumature_blu[1], label='0.1–1 mm/h'),
    mpatches.Patch(color=sfumature_blu[2], label='1–5 mm/h'),
    mpatches.Patch(color=sfumature_blu[3], label='5–15 mm/h'),
    mpatches.Patch(color=sfumature_blu[4], label='>15 mm/h'),
]
plt.legend(handles=legenda, title='Classi di DPR', loc='upper right')

plt.show()

In [1]:
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

# Etichette delle classi (ordinate)
ordine_classi = ['Dry', 'Light', 'Moderate', 'Heavy', 'Intense']

# Bin DPR (assumiamo dpr_tot in mm/h)
dpr_bins = [0, 0.1, 1, 5, 15, np.max(dpr_tot)]
dpr_classi_idx = np.digitize(dpr_tot, dpr_bins) - 1
dpr_classi_labels = [ordine_classi[i] if 0 <= i < len(ordine_classi) else 'Unknown' for i in dpr_classi_idx]

# Conteggi per ciascuna classe
conteggio_dpr = Counter(dpr_classi_labels)
conteggio_pred = Counter(y_pred)

# Frequenze assolute (per ordine)
frequenze_dpr = np.array([conteggio_dpr.get(label, 0) for label in ordine_classi])
frequenze_pred = np.array([conteggio_pred.get(label, 0) for label in ordine_classi])

# ░░░ GRAFICO 1: ASSOLUTO ░░░
x = np.arange(len(ordine_classi))
larghezza = 0.35

plt.figure(figsize=(10, 6))
bar1 = plt.bar(x - larghezza/2, frequenze_dpr, width=larghezza, label='DPR Reale', color='#3399ff')
bar2 = plt.bar(x + larghezza/2, frequenze_pred, width=larghezza, label='Predizione', color='#ff9933')

# Etichette sopra le barre
for rect in bar1 + bar2:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height + 50, str(int(height)), ha='center', va='bottom', fontsize=8)

plt.xticks(x, ordine_classi)
plt.ylabel('Numero di pixel')
plt.title('Distribuzione Assoluta: DPR vs Predizione')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ░░░ GRAFICO 2: NORMALIZZATO (percentuale) ░░░
frequenze_dpr_norm = frequenze_dpr / frequenze_dpr.sum() * 100
frequenze_pred_norm = frequenze_pred / frequenze_pred.sum() * 100

plt.figure(figsize=(10, 6))
bar1 = plt.bar(x - larghezza/2, frequenze_dpr_norm, width=larghezza, label='DPR Reale', color='#3399ff')
bar2 = plt.bar(x + larghezza/2, frequenze_pred_norm, width=larghezza, label='Predizione', color='#ff9933')

# Etichette percentuali
for rect in bar1 + bar2:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2, height + 0.5, f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

plt.xticks(x, ordine_classi)
plt.ylabel('Percentuale di pixel (%)')
plt.title('Distribuzione Percentuale: DPR vs Predizione')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

NameError: name 'dpr_tot' is not defined